# Prediction cutoff and fallback on the current DBSCAN leader

Use the frozen `dbscan_m5_q25__logistic_C1000` model. Evaluate final prediction cutoffs **0.40, 0.45, 0.50, 0.55, 0.60**, each **with and without the nonempty fallback**, on the same 78 validation trials. The fallback selects the highest-probability reason only when none meets the cutoff. These are final prediction cutoffs, distinct from confidence thresholds for accepting pseudo-labels.

No model fitting, pseudo-label generation, feature changes, or Kaggle submission. Save all 10 prediction sets, metrics, per-trial changes, fallback diagnostics, and a selected-policy record referencing the original model. Select by sample-wise F1; ties prefer the current cutoff 0.5 with fallback. Validation tuning remains exploratory.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import f1_score
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
FEATURES=[f"emb_{i}" for i in range(150)]
CUTOFFS=[0.40,0.45,0.50,0.55,0.60]
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_val.csv").is_file() or (p/"data/part1_val.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_val.csv").is_file() else DATA/"data"
OUT=DATA/"experiments/dbscan_prediction_cutoffs"
(OUT/"validation_predictions").mkdir(parents=True,exist_ok=True)
REFERENCE=DATA/"experiments/additional_methods"
record=json.loads((REFERENCE/"submission_manifest.json").read_text())
model_path=REFERENCE/record["model_file"]
model=joblib.load(model_path)
assert model["kind"]=="logistic" and model["features"]==FEATURES and model["reasons"]==REASONS
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
assert len(valid)==78 and valid.ID.is_unique
truth=valid[REASONS].to_numpy(dtype=int)
assert np.isin(truth,[0,1]).all()
X=valid[FEATURES].to_numpy();assert np.isfinite(X).all()
scaled=model["scaler"].transform(X)
probabilities=np.column_stack([est.predict_proba(scaled)[:,list(est.classes_).index(1)] for est in model["estimators"]])
assert probabilities.shape==(78,5) and np.isfinite(probabilities).all()
assert ((probabilities>=0)&(probabilities<=1)).all()
probability_frame=valid[["ID"]].copy()
for j,r in enumerate(REASONS):probability_frame[f"probability_{r}"]=probabilities[:,j]
probability_frame.to_csv(OUT/"validation_probabilities.csv",index=False)
previous=pd.read_csv(REFERENCE/"validation_predictions"/f"{record['experiment']}.csv",float_precision="round_trip")
assert previous.ID.tolist()==valid.ID.tolist()
np.testing.assert_allclose(probabilities,previous[[f"probability_{r}" for r in REASONS]].to_numpy(),rtol=1e-12,atol=1e-12)
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",model_path,REFERENCE/"submission_manifest.json",
                 REFERENCE/record["submission_file"],REFERENCE/"datasets/dbscan_m5_q25.csv"]
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("Frozen model:",record["experiment"])
print("Validation rows:",len(valid),"trials with no actual reasons:",int((truth.sum(axis=1)==0).sum()))


Frozen model: dbscan_m5_q25__logistic_C1000
Validation rows: 78 trials with no actual reasons: 0


In [2]:
def apply_policy(probs,cutoff,fallback):
    prediction=(probs>=cutoff).astype(int)
    empty=prediction.sum(axis=1)==0
    used=empty&bool(fallback)
    rows=np.flatnonzero(used)
    prediction[rows,probs[rows].argmax(axis=1)]=1
    return prediction,empty,used


def trial_f1(actual,predicted):
    numerator=2*(actual*predicted).sum(axis=1)
    denominator=actual.sum(axis=1)+predicted.sum(axis=1)
    return np.divide(numerator,denominator,out=np.zeros(len(actual),dtype=float),where=denominator!=0)

ROWS=[];PREDICTIONS={};CASE_FRAMES=[]
reference_prediction,_,_=apply_policy(probabilities,0.5,True)
reference_trial_f1=trial_f1(truth,reference_prediction)
np.testing.assert_array_equal(reference_prediction,previous[[f"predicted_{r}" for r in REASONS]].to_numpy())
np.testing.assert_allclose(reference_trial_f1.mean(),record["validation_sample_f1"],atol=1e-14)
for cutoff in CUTOFFS:
    for fallback in [True,False]:
        name=f"cutoff_{cutoff:.2f}_fallback_{'on' if fallback else 'off'}".replace(".","p")
        prediction,empty,used=apply_policy(probabilities,cutoff,fallback)
        sample_f1=f1_score(truth,prediction,average="samples",zero_division=0)
        per_trial=trial_f1(truth,prediction)
        np.testing.assert_allclose(per_trial.mean(),sample_f1,atol=1e-14)
        assert (prediction.sum(axis=1)>0).all() if fallback else not used.any()
        row={"experiment":name,"cutoff":cutoff,"fallback":fallback,"sample_f1":sample_f1,
             "macro_f1":f1_score(truth,prediction,average="macro",zero_division=0),
             "empty_before_fallback":int(empty.sum()),"fallback_trials":int(used.sum()),
             "empty_predictions":int((prediction.sum(axis=1)==0).sum()),
             "predicted_positive_labels":int(prediction.sum()),
             "changed_trials_vs_current":int((prediction!=reference_prediction).any(axis=1).sum()),
             "improved_trials_vs_current":int((per_trial>reference_trial_f1+1e-12).sum()),
             "worsened_trials_vs_current":int((per_trial<reference_trial_f1-1e-12).sum()),
             "delta_vs_current":sample_f1-reference_trial_f1.mean(),
             "is_current":bool(cutoff==0.5 and fallback),
             **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,prediction,average=None,zero_division=0)))}
        ROWS.append(row);PREDICTIONS[name]=prediction
        frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
        for j,r in enumerate(REASONS):
            frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=prediction[:,j]
        frame["empty_before_fallback"]=empty;frame["used_fallback"]=used
        frame["empty_prediction"]=prediction.sum(axis=1)==0
        frame["trial_f1"]=per_trial;frame["current_trial_f1"]=reference_trial_f1
        frame["delta_vs_current"]=per_trial-reference_trial_f1
        frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
        cases=frame[["ID","trial_f1","current_trial_f1","delta_vs_current","used_fallback","empty_prediction"]].copy()
        cases.insert(0,"experiment",name);CASE_FRAMES.append(cases)
comparison=pd.DataFrame(ROWS)
comparison.to_csv(OUT/"comparison.csv",index=False)
pd.concat(CASE_FRAMES,ignore_index=True).to_csv(OUT/"per_trial_comparison.csv",index=False)
assert len(comparison)==10
# Larger cutoffs cannot add threshold-selected labels or reduce the number of empty threshold predictions.
without=comparison.query("fallback==False").sort_values("cutoff")
assert (np.diff(without.predicted_positive_labels)<=0).all()
assert (np.diff(without.empty_before_fallback)>=0).all()
ranked=comparison.assign(distance_from_current=lambda d:abs(d.cutoff-0.5)).sort_values(
    ["sample_f1","is_current","distance_from_current","fallback","cutoff"],ascending=[False,False,True,False,True],kind="stable")
best=ranked.iloc[0]
display(comparison[["cutoff","fallback","sample_f1","macro_f1","empty_predictions","fallback_trials","delta_vs_current"]].round(6))
print("Selected:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)


,cutoff,fallback,sample_f1,macro_f1,empty_predictions,fallback_trials,delta_vs_current
0,0.40,True,0.910256,0.811421,0,4,0.000000
1,0.40,False,0.876068,0.804787,4,0,-0.034188
2,0.45,True,0.910256,0.811421,0,4,0.000000
3,0.45,False,0.876068,0.804787,4,0,-0.034188
4,0.50,True,0.910256,0.811421,0,4,0.000000
5,0.50,False,0.876068,0.804787,4,0,-0.034188
6,0.55,True,0.910256,0.811421,0,4,0.000000
7,0.55,False,0.876068,0.804787,4,0,-0.034188
8,0.60,True,0.910256,0.811421,0,4,0.000000
9,0.60,False,0.876068,0.804787,4,0,-0.034188


Selected: cutoff_0p50_fallback_on sample F1: 0.9102564102564102 macro F1: 0.811421460284173


In [3]:
fallback_rows=[];fallback_cases=[]
for cutoff in CUTOFFS:
    pred_off,empty,_=apply_policy(probabilities,cutoff,False)
    pred_on,_,used=apply_policy(probabilities,cutoff,True)
    np.testing.assert_array_equal(pred_on[~empty],pred_off[~empty])
    assert (pred_on[empty].sum(axis=1)==1).all()
    before=trial_f1(truth,pred_off);after=trial_f1(truth,pred_on)
    gain=after-before
    assert (gain>=-1e-12).all() # Adding one label to an empty prediction can only tie or improve sample F1 here.
    fallback_rows.append({"cutoff":cutoff,"affected_trials":int(empty.sum()),
                          "helped_trials":int((gain>1e-12).sum()),"unchanged_trials":int((empty&(gain<=1e-12)).sum()),
                          "sample_f1_gain":float(gain.mean())})
    for i in np.flatnonzero(empty):
        j=int(probabilities[i].argmax())
        fallback_cases.append({"cutoff":cutoff,"ID":valid.ID.iloc[i],"fallback_reason":REASONS[j],
                               "maximum_probability":float(probabilities[i,j]),"reason_correct":bool(truth[i,j]),
                               "actual_reasons":" ".join(r for r,yes in zip(REASONS,truth[i]) if yes),
                               "trial_f1_without_fallback":float(before[i]),"trial_f1_with_fallback":float(after[i]),
                               "trial_f1_gain":float(gain[i])})
fallback_comparison=pd.DataFrame(fallback_rows)
fallback_comparison.to_csv(OUT/"fallback_comparison.csv",index=False)
pd.DataFrame(fallback_cases).to_csv(OUT/"fallback_cases.csv",index=False)
# Reload every saved prediction and independently recompute metrics.
for row in comparison.itertuples(index=False):
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    predicted=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_array_equal(predicted,PREDICTIONS[row.experiment])
    np.testing.assert_allclose(f1_score(truth,predicted,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predicted,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest
policy={"model_file":str(model_path.relative_to(DATA)),"model_sha256":hashlib.sha256(model_path.read_bytes()).hexdigest(),
        "cutoff":float(best.cutoff),"fallback":bool(best.fallback),"reasons":REASONS,"features":FEATURES,
        "validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
        "experiment":best.experiment,"model_retrained":False,"kaggle_score":None,"kaggle_status":"not submitted"}
(OUT/"selected_policy.json").write_text(json.dumps(policy,indent=2)+"\n")
settings={"cutoffs":CUTOFFS,"fallback_options":[True,False],"reference_cutoff":0.5,"reference_fallback":True,
          "source_model":record["experiment"],"source_model_file":str(model_path.relative_to(DATA)),
          "validation_rows":len(valid),"validation_used_for_fit":False,"model_retrained":False,
          "primary_metric":"sample-wise F1","zero_division":0,"selected_policy":policy,
          "protected_input_sha256":protected_hashes,
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},
          "selection":"sample-wise F1; ties prefer current policy, then closest cutoff, then fallback enabled",
          "kaggle_submission_used":False}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
display(fallback_comparison.round(6))
print("All 10 saved prediction sets reproduce their metrics; original-model probabilities and protected hashes match.")


,cutoff,affected_trials,helped_trials,unchanged_trials,sample_f1_gain
0,0.40,4,3,1,0.034188
1,0.45,4,3,1,0.034188
2,0.50,4,3,1,0.034188
3,0.55,4,3,1,0.034188
4,0.60,4,3,1,0.034188


All 10 saved prediction sets reproduce their metrics; original-model probabilities and protected hashes match.


In [4]:
lines=["# Current DBSCAN leader: prediction cutoff and fallback","",
       f"Frozen model: `{record['experiment']}`. No training, feature changes, pseudo-label changes, or Kaggle submission. "
       "The same 78 validation trials were evaluated at final cutoffs 0.40, 0.45, 0.50, 0.55, 0.60, with and without fallback.","",
       "| Cutoff | Sample F1 with fallback | Sample F1 without | Macro F1 with | Macro F1 without | Empty before fallback |",
       "|---:|---:|---:|---:|---:|---:|"]
for cutoff in CUTOFFS:
    on=comparison.loc[(comparison.cutoff==cutoff)&comparison.fallback].iloc[0]
    off=comparison.loc[(comparison.cutoff==cutoff)&~comparison.fallback].iloc[0]
    lines.append(f"| {cutoff:.2f} | {on.sample_f1:.6f} | {off.sample_f1:.6f} | {on.macro_f1:.6f} | {off.macro_f1:.6f} | {on.empty_before_fallback} |")
lines.extend(["",f"Selected policy: cutoff **{best.cutoff:.2f}**, fallback **{'enabled' if best.fallback else 'disabled'}**. "
              f"Sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**. "
              "Ties prefer the existing cutoff 0.50 with fallback.","",
              "## Fallback contribution","",
              "| Cutoff | Trials needing fallback | Helped trials | Unchanged trials | Sample F1 gain |",
              "|---:|---:|---:|---:|---:|"])
for row in fallback_comparison.itertuples(index=False):
    lines.append(f"| {row.cutoff:.2f} | {row.affected_trials} | {row.helped_trials} | {row.unchanged_trials} | {row.sample_f1_gain:.6f} |")
lines.extend(["","Without fallback, trials with no selected labels remain all-zero predictions. "
              "With sample-wise F1 and zero_division=0, selecting a reason for such a trial either leaves its F1 at zero "
              "or increases it if the selected reason is correct. Thus fallback cannot lower sample-wise F1 for these "
              "empty predictions under this scoring convention, although it can change per-label precision and macro F1.","",
              "## Saved artifacts and checks","",
              "`validation_probabilities.csv` freezes the single probability matrix used by all settings. "
              "`validation_predictions/` saves all ten label sets and per-trial scores. `comparison.csv` includes "
              "sample, macro, per-reason F1, empty predictions, and changes versus the current policy. "
              "`fallback_comparison.csv`, `fallback_cases.csv`, and `per_trial_comparison.csv` explain the changes. "
              "`selected_policy.json` references the original model and chosen postprocessing; no model was retrained.","",
              "The original cutoff/fallback reproduces the previous predictions and sample F1. All saved prediction "
              "sets were reloaded and independently rescored, threshold monotonicity and fallback invariants passed, "
              "and protected inputs and model artifacts retained their hashes.","",
              "Selection uses the repeatedly consulted validation set and does not establish independent test improvement. "
              "No Kaggle submission, commit, or tag was made."])
summary="\n".join(lines)+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(summary[:3200])


# Current DBSCAN leader: prediction cutoff and fallback

Frozen model: `dbscan_m5_q25__logistic_C1000`. No training, feature changes, pseudo-label changes, or Kaggle submission. The same 78 validation trials were evaluated at final cutoffs 0.40, 0.45, 0.50, 0.55, 0.60, with and without fallback.

| Cutoff | Sample F1 with fallback | Sample F1 without | Macro F1 with | Macro F1 without | Empty before fallback |
|---:|---:|---:|---:|---:|---:|
| 0.40 | 0.910256 | 0.876068 | 0.811421 | 0.804787 | 4 |
| 0.45 | 0.910256 | 0.876068 | 0.811421 | 0.804787 | 4 |
| 0.50 | 0.910256 | 0.876068 | 0.811421 | 0.804787 | 4 |
| 0.55 | 0.910256 | 0.876068 | 0.811421 | 0.804787 | 4 |
| 0.60 | 0.910256 | 0.876068 | 0.811421 | 0.804787 | 4 |

Selected policy: cutoff **0.50**, fallback **enabled**. Sample F1 **0.910256**, macro F1 **0.811421**. Ties prefer the existing cutoff 0.50 with fallback.

## Fallback contribution

| Cutoff | Trials needing fallback | Helped trials | Unchanged trials | Sample F1 gain